# Phase 3C: Teacher A Training Readiness

Notebook này là cổng kỹ thuật cuối trước full training Teacher A.

Teacher A:

`CLIP ViT-B/32 + adapter 768→256 + shared Transformer decoder`

Notebook thực hiện:

1. verify Phase 03B bundle;
2. dựng Train/Validation indexes từ Phase 1;
3. dựng image-balanced sampler và stateless augmentation;
4. strict-load Teacher A;
5. dựng staged optimizer/scheduler/GradScaler;
6. chạy one-batch overfit và loại bỏ weights;
7. chạy một optimizer-step FP16 và phục hồi initial state;
8. kiểm tra exact resume;
9. chạy discarded throughput pilot 20 optimizer steps;
10. tạo `PHASE3_TEACHER_A_FULL_TRAINING_READY.json`.

Held-out Test luôn `SEALED`. Notebook chưa chạy full training.

## Kaggle inputs

- Phase 1 Formal Output.
- Phase 2A Shared Pipeline.
- Phase 3A Teacher Pool Entry Gate.
- Phase 3B Teacher A Preflight.
- MS COCO 2014 images.
- Offline `openai/clip-vit-base-patch32`.

## Kaggle settings

- GPU T4.
- Internet Off.
- Chạy từng cell theo thứ tự.


## Cell 1 — Environment và verify Phase 03B

### Công việc

- khóa seed 42 và math SDPA;
- tìm đúng một Phase 03B READY marker;
- verify toàn bộ manifest rows;
- xác minh source, architecture, preprocessing và training contract;
- chưa đọc data hoặc model weights.

### Thời gian dự kiến

15–120 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Phase 03B                 : VERIFIED
Training contract         : LOCKED
Test Set status           : SEALED
Progress                  : 1 / 10 cells passed
```


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = (
    KAGGLE_WORKING_ROOT
    / ""
    / "phase3_teacher_a_training_readiness"
)
CONFIG_DIR = OUTPUT_ROOT / "configs"
AUDIT_DIR = OUTPUT_ROOT / "audits"
LOG_DIR = OUTPUT_ROOT / "logs"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    CONFIG_DIR,
    AUDIT_DIR,
    LOG_DIR,
    CHECKPOINT_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        while True:
            chunk = source.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    candidate = (root / relative_path).resolve()
    assert candidate.is_relative_to(root)
    return candidate


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


assert KAGGLE_INPUT_ROOT.is_dir()
assert torch.cuda.is_available(), "Hãy bật GPU T4."
set_deterministic(SEED)
device = torch.device("cuda")

ready_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE3_TEACHER_A_PREFLIGHT_READY.json"
    )
)
assert len(ready_matches) == 1, (
    f"Cần đúng một Phase 03B input, thấy {len(ready_matches)}."
)
PHASE3B_READY_PATH = ready_matches[0].resolve()
PHASE3B_ROOT = PHASE3B_READY_PATH.parent
PHASE3B_READY = read_json(PHASE3B_READY_PATH)
assert PHASE3B_READY["status"] == "ready"
assert PHASE3B_READY["teacher_id"] == "A"
assert PHASE3B_READY["preflight_cells_passed"] == 11
assert PHASE3B_READY["training_started"] is False
assert PHASE3B_READY["test_set_status"] == "sealed"
assert PHASE3B_READY["test_set_used"] is False

manifest_record = PHASE3B_READY["manifests"]["csv"]
manifest_path = resolve_inside(
    PHASE3B_ROOT,
    manifest_record["relative_path"],
)
assert sha256_file(manifest_path) == manifest_record["sha256"]
with open(
    manifest_path,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    phase3b_manifest_rows = list(csv.DictReader(source))
for row in phase3b_manifest_rows:
    artifact = resolve_inside(PHASE3B_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

TRAINING_CONTRACT_PATH = resolve_inside(
    PHASE3B_ROOT,
    PHASE3B_READY["training_contract"]["relative_path"],
)
assert sha256_file(TRAINING_CONTRACT_PATH) == (
    PHASE3B_READY["training_contract"]["sha256"]
)
TRAINING_CONTRACT = read_json(TRAINING_CONTRACT_PATH)
assert TRAINING_CONTRACT["status"] == "locked"
assert TRAINING_CONTRACT["data_policy"]["test_set_status"] == "sealed"

PHASE3B_LIGHT_CHECKPOINT = (
    PHASE3B_ROOT
    / "checkpoints"
    / "teacher_a_adapter_decoder_initial_seed42.pt"
)
assert PHASE3B_LIGHT_CHECKPOINT.is_file()

audit_path = AUDIT_DIR / "phase3c_cell01_phase3b_environment.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase3b_manifest_rows": len(phase3b_manifest_rows),
        "phase3b_ready_sha256": sha256_file(PHASE3B_READY_PATH),
        "training_contract_sha256": sha256_file(
            TRAINING_CONTRACT_PATH
        ),
        "gpu": torch.cuda.get_device_name(0),
        "strict_deterministic": True,
        "data_read": False,
        "model_loaded": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — CLIP ViT-B/32")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Phase 03B                 : VERIFIED")
print("Manifest rows             :", len(phase3b_manifest_rows), "PASSED")
print("Training contract         : LOCKED")
print("Strict deterministic      : True")
print("Data read                 : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 10 cells passed")
print("Next                      : load Phase 1 Train/Validation indexes")

PHASE 3C CELL 1 STATUS
Status                    : PASSED
Teacher                   : A — CLIP ViT-B/32
GPU                       : Tesla T4
Phase 03B                 : VERIFIED
Manifest rows             : 15 PASSED
Training contract         : LOCKED
Strict deterministic      : True
Data read                 : False
Training started          : False
Test Set status           : SEALED
Progress                  : 1 / 10 cells passed
Next                      : load Phase 1 Train/Validation indexes


## Cell 2 — Phase 1 indexes và MS COCO image paths

### Công việc

- verify Phase 1 READY marker;
- đọc Train và Validation token IDs;
- dựng map 113.287 train images và 5.000 validation images;
- giữ 5.000 Test IDs chỉ để kiểm tra overlap;
- không mở Test Parquet.

### Thời gian dự kiến

30–120 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Training captions         : 566747
Training images           : 113287
Validation captions       : 25010
Validation images         : 5000
Test overlap              : 0
Progress                  : 2 / 10 cells passed
```


In [2]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow.parquet as pq

required = [
    "KAGGLE_INPUT_ROOT",
    "AUDIT_DIR",
    "read_json",
    "write_json",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1: {missing}"


def exactly_one(root, pattern):
    matches = sorted(Path(root).rglob(pattern))
    assert len(matches) == 1, (
        f"Cần đúng một {pattern}, thấy {len(matches)}."
    )
    return matches[0].resolve()


phase1_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE1_FORMAL_DELIVERABLES_READY.json"
    )
)
assert len(phase1_matches) == 1
PHASE1_ROOT = phase1_matches[0].resolve().parent
TRAIN_PARQUET = exactly_one(PHASE1_ROOT, "mscoco_train.parquet")
VALIDATION_PARQUET = exactly_one(
    PHASE1_ROOT, "mscoco_validation.parquet"
)
IMAGE_MANIFEST_PARQUET = exactly_one(
    PHASE1_ROOT, "mscoco_image_manifest.parquet"
)
TEST_PARQUET = exactly_one(PHASE1_ROOT, "mscoco_test.parquet")

train_table = pq.read_table(
    TRAIN_PARQUET,
    columns=["image_id", "token_ids"],
)
validation_table = pq.read_table(
    VALIDATION_PARQUET,
    columns=["image_id", "token_ids"],
)
manifest_table = pq.read_table(
    IMAGE_MANIFEST_PARQUET,
    columns=[
        "image_id",
        "experiment_split",
        "source_partition",
        "image_path",
    ],
)


def build_leaf(partition, prefix):
    matches = []
    for directory in KAGGLE_INPUT_ROOT.rglob(partition):
        if directory.is_dir() and next(
            directory.glob(prefix + "*.jpg"),
            None,
        ) is not None:
            matches.append(directory.resolve())
    assert len(matches) == 1, (
        f"Cần đúng một {partition} leaf, thấy {matches}."
    )
    return matches[0]


TRAIN_LEAF = build_leaf("train2014", "COCO_train2014_")
VAL_LEAF = build_leaf("val2014", "COCO_val2014_")
manifest_rows = manifest_table.to_pylist()
TRAIN_PATH_BY_ID = {}
VALIDATION_PATH_BY_ID = {}
TEST_IDS = set()
for row in manifest_rows:
    image_id = int(row["image_id"])
    split = row["experiment_split"]
    partition = row["source_partition"]
    file_name = Path(row["image_path"]).name
    if split == "train":
        leaf = TRAIN_LEAF if partition == "train2014" else VAL_LEAF
        image_path = leaf / file_name
        assert image_path.is_file()
        TRAIN_PATH_BY_ID[image_id] = image_path
    elif split == "validation":
        assert partition == "val2014"
        image_path = VAL_LEAF / file_name
        assert image_path.is_file()
        VALIDATION_PATH_BY_ID[image_id] = image_path
    elif split == "held_out_test":
        TEST_IDS.add(image_id)

TRAIN_CAPTIONS_BY_IMAGE = {}
for image_id, tokens in zip(
    train_table["image_id"].to_pylist(),
    train_table["token_ids"].to_pylist(),
):
    TRAIN_CAPTIONS_BY_IMAGE.setdefault(
        int(image_id), []
    ).append([int(token) for token in tokens])
VALIDATION_RECORDS = [
    {
        "image_id": int(image_id),
        "token_ids": [int(token) for token in tokens],
    }
    for image_id, tokens in zip(
        validation_table["image_id"].to_pylist(),
        validation_table["token_ids"].to_pylist(),
    )
]

assert train_table.num_rows == 566747
assert validation_table.num_rows == 25010
assert len(TRAIN_PATH_BY_ID) == 113287
assert len(TRAIN_CAPTIONS_BY_IMAGE) == 113287
assert len(VALIDATION_PATH_BY_ID) == 5000
assert len(TEST_IDS) == 5000
assert not (set(TRAIN_PATH_BY_ID) & TEST_IDS)
assert not (set(VALIDATION_PATH_BY_ID) & TEST_IDS)
assert not (
    set(TRAIN_PATH_BY_ID) & set(VALIDATION_PATH_BY_ID)
)

audit_path = AUDIT_DIR / "phase3c_cell02_data_indexes.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "training_caption_rows": train_table.num_rows,
        "training_unique_images": len(TRAIN_PATH_BY_ID),
        "validation_caption_rows": validation_table.num_rows,
        "validation_unique_images": len(VALIDATION_PATH_BY_ID),
        "held_out_test_ids": len(TEST_IDS),
        "all_membership_overlaps": 0,
        "caption_text_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Training captions         :", train_table.num_rows)
print("Training images           :", len(TRAIN_PATH_BY_ID))
print("Validation captions       :", validation_table.num_rows)
print("Validation images         :", len(VALIDATION_PATH_BY_ID))
print("Held-out Test IDs         :", len(TEST_IDS))
print("Membership overlap        : 0")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 10 cells passed")
print("Next                      : build deterministic DataLoader")

PHASE 3C CELL 2 STATUS
Status                    : PASSED
Training captions         : 566747
Training images           : 113287
Validation captions       : 25010
Validation images         : 5000
Held-out Test IDs         : 5000
Membership overlap        : 0
Caption text read         : False
Test Parquet opened       : False
Test Set status           : SEALED
Progress                  : 2 / 10 cells passed
Next                      : build deterministic DataLoader


## Cell 3 — Image-balanced sampler và stateless augmentation

### Công việc

- chọn đúng một caption cho mỗi ảnh mỗi epoch;
- xoay caption theo epoch;
- shuffle ảnh bằng seed cố định;
- augmentation crop 0,8–1,0 và flip 0,5;
- CLIP normalization;
- kiểm tra hai Dataset cho tensor giống tuyệt đối.

### Thời gian dự kiến

5–30 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Rows per epoch            : 113287
Physical/effective batch  : 16 / 64
Train repeat error        : 0.0
Test overlap              : 0
Progress                  : 3 / 10 cells passed
```


In [3]:
CELL_STARTED_AT = time.perf_counter()
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

required = [
    "TRAIN_PATH_BY_ID",
    "TRAIN_CAPTIONS_BY_IMAGE",
    "TEST_IDS",
    "SEED",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–2: {missing}"

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
PHYSICAL_BATCH_SIZE = 16
EFFECTIVE_BATCH_SIZE = 64
GRADIENT_ACCUMULATION = 4

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            224,
            scale=(0.80, 1.00),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.RandomHorizontalFlip(p=0.50),
        transforms.ToTensor(),
        transforms.Normalize(CLIP_MEAN, CLIP_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.Resize(
            224,
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(CLIP_MEAN, CLIP_STD),
    ]
)


def sample_seed(base_seed, epoch, image_id, position):
    value = (
        int(base_seed) * 1000003
        + int(epoch) * 9176
        + int(image_id) * 37
        + int(position) * 101
    )
    return value % (2**31 - 1)


class ImageBalancedTeacherDataset(Dataset):
    def __init__(self, base_seed, augment=True):
        self.base_seed = int(base_seed)
        self.augment = bool(augment)
        self.epoch = 0
        self.image_ids = sorted(TRAIN_PATH_BY_ID)
        self.order = []
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.order = list(self.image_ids)
        random.Random(
            self.base_seed + self.epoch
        ).shuffle(self.order)

    def __len__(self):
        return len(self.order)

    def __getitem__(self, position):
        image_id = self.order[position]
        assert image_id not in TEST_IDS
        captions = TRAIN_CAPTIONS_BY_IMAGE[image_id]
        token_ids = captions[self.epoch % len(captions)]
        with Image.open(TRAIN_PATH_BY_ID[image_id]) as image:
            image = image.convert("RGB")
            if self.augment:
                seed = sample_seed(
                    self.base_seed,
                    self.epoch,
                    image_id,
                    position,
                )
                with torch.random.fork_rng(devices=[]):
                    torch.manual_seed(seed)
                    pixel_values = train_transform(image)
            else:
                pixel_values = eval_transform(image)
        return {
            "image_id": image_id,
            "pixel_values": pixel_values,
            "token_ids": token_ids,
        }


def collate_teacher(records):
    maximum_length = max(
        len(record["token_ids"]) for record in records
    )
    full_ids = torch.zeros(
        len(records),
        maximum_length,
        dtype=torch.long,
    )
    for index, record in enumerate(records):
        sequence = torch.tensor(
            record["token_ids"],
            dtype=torch.long,
        )
        full_ids[index, : sequence.numel()] = sequence
    return {
        "image_ids": torch.tensor(
            [record["image_id"] for record in records],
            dtype=torch.long,
        ),
        "pixel_values": torch.stack(
            [record["pixel_values"] for record in records]
        ),
        "decoder_input_ids": full_ids[:, :-1],
        "decoder_attention_mask": full_ids[:, :-1].ne(0),
        "labels": full_ids[:, 1:],
    }


def make_loader(dataset, start_position=0, workers=2):
    indices = list(range(start_position, len(dataset)))
    subset = torch.utils.data.Subset(dataset, indices)
    return DataLoader(
        subset,
        batch_size=PHYSICAL_BATCH_SIZE,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        persistent_workers=workers > 0,
        collate_fn=collate_teacher,
    )


TRAIN_DATASET = ImageBalancedTeacherDataset(SEED, augment=True)
TRAIN_DATASET_DUPLICATE = ImageBalancedTeacherDataset(
    SEED, augment=True
)
first_a = TRAIN_DATASET[0]
first_b = TRAIN_DATASET_DUPLICATE[0]
repeat_error = (
    first_a["pixel_values"] - first_b["pixel_values"]
).abs().max().item()
assert repeat_error == 0.0
assert first_a["image_id"] == first_b["image_id"]
assert first_a["token_ids"] == first_b["token_ids"]
assert len(TRAIN_DATASET) == 113287
assert PHYSICAL_BATCH_SIZE * GRADIENT_ACCUMULATION == (
    EFFECTIVE_BATCH_SIZE
)

smoke_loader = make_loader(TRAIN_DATASET)
smoke_batch = next(iter(smoke_loader))
assert list(smoke_batch["pixel_values"].shape) == [
    16, 3, 224, 224
]
assert len(set(smoke_batch["image_ids"].tolist())) == 16
assert not (set(smoke_batch["image_ids"].tolist()) & TEST_IDS)

contract_path = CONFIG_DIR / "teacher_a_training_data_contract.json"
write_json(
    contract_path,
    {
        "contract_version": "phase3_teacher_a_training_data_v1",
        "status": "locked",
        "rows_per_epoch": 113287,
        "captions_per_image_per_epoch": 1,
        "caption_rotation": "epoch_mod_caption_count",
        "stateless_augmentation": True,
        "random_crop_scale": [0.8, 1.0],
        "horizontal_flip_probability": 0.5,
        "input_size": [224, 224],
        "normalization_mean": list(CLIP_MEAN),
        "normalization_std": list(CLIP_STD),
        "physical_batch_size": PHYSICAL_BATCH_SIZE,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
audit_path = AUDIT_DIR / "phase3c_cell03_dataloader.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "rows_per_epoch": len(TRAIN_DATASET),
        "smoke_batch_shape": list(
            smoke_batch["pixel_values"].shape
        ),
        "unique_images_in_batch": 16,
        "repeat_maximum_error": repeat_error,
        "test_overlap": 0,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

del smoke_loader
print("=" * 72)
print("PHASE 3C CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Rows per epoch            :", len(TRAIN_DATASET))
print("Captions/image/epoch      : 1")
print("Physical/effective batch  : 16 / 64")
print("Gradient accumulation     : 4")
print("Train tensor              :", list(smoke_batch["pixel_values"].shape))
print("Train repeat error        :", repeat_error)
print("Stateless augmentation    : LOCKED")
print("Test overlap              : 0")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 10 cells passed")
print("Next                      : reconstruct Teacher A")

PHASE 3C CELL 3 STATUS
Status                    : PASSED
Rows per epoch            : 113287
Captions/image/epoch      : 1
Physical/effective batch  : 16 / 64
Gradient accumulation     : 4
Train tensor              : [16, 3, 224, 224]
Train repeat error        : 0.0
Stateless augmentation    : LOCKED
Test overlap              : 0
Training started          : False
Test Set status           : SEALED
Progress                  : 3 / 10 cells passed
Next                      : reconstruct Teacher A


## Cell 4 — Reconstruct Teacher A và strict-load initial state

### Công việc

- tìm offline CLIP ViT-B/32;
- dựng vision encoder, adapter và shared decoder;
- strict-load adapter/decoder checkpoint từ Phase 03B;
- kiểm tra forward trên batch thật;
- không download network.

### Thời gian dự kiến

20–90 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Patch tokens              : [2, 49, 768]
Visual memory             : [2, 49, 256]
Logits                    : [2, T, 10080]
Strict load               : PASSED
Progress                  : 4 / 10 cells passed
```


In [4]:
CELL_STARTED_AT = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel

required = [
    "KAGGLE_INPUT_ROOT",
    "PHASE3B_LIGHT_CHECKPOINT",
    "TRAIN_DATASET",
    "collate_teacher",
    "device",
    "SEED",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–3: {missing}"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
clip_candidates = []
for config_path in KAGGLE_INPUT_ROOT.rglob("config.json"):
    try:
        payload = read_json(config_path)
    except Exception:
        continue
    vision = payload.get("vision_config", payload)
    if (
        int(vision.get("hidden_size", -1)) == 768
        and int(vision.get("patch_size", -1)) == 32
        and int(vision.get("image_size", -1)) == 224
    ):
        weights = sorted(
            list(config_path.parent.glob("*.safetensors"))
            + list(config_path.parent.glob("pytorch_model*.bin"))
        )
        if weights:
            clip_candidates.append(config_path.parent.resolve())
assert len(clip_candidates) == 1, (
    f"Cần đúng một offline CLIP input, thấy {clip_candidates}."
)
CLIP_SOURCE_DIR = clip_candidates[0]

DECODER_CONFIG = {
    "vocabulary_size": 10080,
    "pad_id": 0,
    "bos_id": 1,
    "eos_id": 2,
    "unk_id": 3,
    "d_model": 256,
    "nhead": 8,
    "head_dimension": 32,
    "num_layers": 3,
    "dim_feedforward": 1024,
    "dropout": 0.1,
    "activation": "gelu",
    "maximum_decoder_steps": 31,
    "memory_dimension": 256,
    "norm_first": True,
    "batch_first": True,
    "tie_input_output_embeddings": True,
}


class ClipVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

    def forward(self, patch_tokens):
        return self.output_norm(
            self.projection(patch_tokens)
            + self.position_embedding
        )


class SharedCaptionTransformerDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.pad_id = config["pad_id"]
        self.d_model = config["d_model"]
        self.maximum_decoder_steps = config[
            "maximum_decoder_steps"
        ]
        self.token_embedding = nn.Embedding(
            config["vocabulary_size"],
            config["d_model"],
            padding_idx=config["pad_id"],
        )
        self.position_embedding = nn.Embedding(
            config["maximum_decoder_steps"],
            config["d_model"],
        )
        self.embedding_dropout = nn.Dropout(config["dropout"])
        layer = nn.TransformerDecoderLayer(
            d_model=config["d_model"],
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer,
            num_layers=config["num_layers"],
        )
        self.final_norm = nn.LayerNorm(config["d_model"])
        self.output_bias = nn.Parameter(
            torch.zeros(config["vocabulary_size"])
        )

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask,
    ):
        batch_size, sequence_length = decoder_input_ids.shape
        positions = torch.arange(
            sequence_length,
            device=decoder_input_ids.device,
        ).unsqueeze(0).expand(batch_size, -1)
        target = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=decoder_input_ids.device,
            ),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            target,
            visual_memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~decoder_attention_mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden,
            self.token_embedding.weight,
            self.output_bias,
        )


full_clip = CLIPModel.from_pretrained(
    CLIP_SOURCE_DIR,
    local_files_only=True,
).to(device)
CLIP_ENCODER = full_clip.vision_model
del full_clip
VISUAL_ADAPTER = ClipVisualAdapter().to(device)
SHARED_DECODER = SharedCaptionTransformerDecoder(
    DECODER_CONFIG
).to(device)

try:
    from torch.torch_version import TorchVersion
    context = torch.serialization.safe_globals([TorchVersion])
except Exception:
    from contextlib import nullcontext
    context = nullcontext()
with context:
    LIGHT_CHECKPOINT = torch.load(
        PHASE3B_LIGHT_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
assert LIGHT_CHECKPOINT["decoder_config"] == DECODER_CONFIG
adapter_load = VISUAL_ADAPTER.load_state_dict(
    LIGHT_CHECKPOINT["adapter_state_dict"],
    strict=True,
)
decoder_load = SHARED_DECODER.load_state_dict(
    LIGHT_CHECKPOINT["decoder_state_dict"],
    strict=True,
)
assert not adapter_load.missing_keys and not adapter_load.unexpected_keys
assert not decoder_load.missing_keys and not decoder_load.unexpected_keys

smoke_records = [TRAIN_DATASET[index] for index in range(2)]
model_batch = collate_teacher(smoke_records)
pixel_values = model_batch["pixel_values"].to(device)
decoder_input_ids = model_batch["decoder_input_ids"].to(device)
decoder_mask = model_batch["decoder_attention_mask"].to(device)
CLIP_ENCODER.eval()
VISUAL_ADAPTER.eval()
SHARED_DECODER.eval()
with torch.inference_mode():
    patches = CLIP_ENCODER(
        pixel_values=pixel_values
    ).last_hidden_state[:, 1:, :]
    memory = VISUAL_ADAPTER(patches)
    logits = SHARED_DECODER(
        decoder_input_ids,
        memory,
        decoder_mask,
    )
assert list(patches.shape) == [2, 49, 768]
assert list(memory.shape) == [2, 49, 256]
assert logits.shape[-1] == 10080
assert torch.isfinite(logits).all()

audit_path = AUDIT_DIR / "phase3c_cell04_model_load.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "clip_source": str(CLIP_SOURCE_DIR),
        "light_checkpoint_sha256": sha256_file(
            PHASE3B_LIGHT_CHECKPOINT
        ),
        "patch_shape": list(patches.shape),
        "memory_shape": list(memory.shape),
        "logits_shape": list(logits.shape),
        "strict_load": True,
        "forward_finite": True,
        "network_download": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Encoder                   : CLIP ViT-B/32")
print("Patch tokens              :", list(patches.shape))
print("Visual memory             :", list(memory.shape))
print("Logits                    :", list(logits.shape))
print("Strict load               : PASSED")
print("Forward finite            : True")
print("Network download          : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 10 cells passed")
print("Next                      : build staged optimizer state")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /kaggle/input/models/nguynthnhdan/clip-vit-base-patch32-offline/pytorch/transformers-pytorch/1/openai_clip-vit-base-patch32_3d74acf9
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PHASE 3C CELL 4 STATUS
Status                    : PASSED
Encoder                   : CLIP ViT-B/32
Patch tokens              : [2, 49, 768]
Visual memory             : [2, 49, 256]
Logits                    : [2, 11, 10080]
Strict load               : PASSED
Forward finite            : True
Network download          : False
Training started          : False
Test Set status           : SEALED
Progress                  : 4 / 10 cells passed
Next                      : build staged optimizer state


## Cell 5 — Staged optimizer, scheduler và GradScaler

### Công việc

- epoch 1–5 đóng băng CLIP encoder;
- epoch 6–20 chỉ mở hai encoder blocks cuối và post-LayerNorm;
- tạo AdamW parameter groups;
- khóa warmup 5%, cosine và FP16 GradScaler;
- chưa thực hiện optimizer step.

### Thời gian dự kiến

Dưới 5 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Stage 1 encoder trainable : 0
Effective batch           : 64
Total optimizer steps     : 35420
Optimizer state           : EMPTY
Progress                  : 5 / 10 cells passed
```


In [5]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "CLIP_ENCODER",
    "VISUAL_ADAPTER",
    "SHARED_DECODER",
    "TRAINING_CONTRACT",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–4: {missing}"

OPTIMIZER_STEPS_PER_EPOCH = math.ceil(
    113287 / EFFECTIVE_BATCH_SIZE
)
TOTAL_OPTIMIZER_STEPS = OPTIMIZER_STEPS_PER_EPOCH * 20
WARMUP_STEPS = round(TOTAL_OPTIMIZER_STEPS * 0.05)


def encoder_candidate_parameters():
    parameters = []
    for layer in CLIP_ENCODER.encoder.layers[-2:]:
        parameters.extend(list(layer.parameters()))
    parameters.extend(list(CLIP_ENCODER.post_layernorm.parameters()))
    unique = []
    seen = set()
    for parameter in parameters:
        if id(parameter) not in seen:
            unique.append(parameter)
            seen.add(id(parameter))
    return unique


ENCODER_CANDIDATE_PARAMETERS = encoder_candidate_parameters()


def set_training_stage(epoch):
    for parameter in CLIP_ENCODER.parameters():
        parameter.requires_grad = False
    if int(epoch) >= 6:
        for parameter in ENCODER_CANDIDATE_PARAMETERS:
            parameter.requires_grad = True
        stage = "partial_unfreeze"
    else:
        stage = "frozen_encoder"
    return stage


def lr_factor(step):
    step = int(step)
    if step < WARMUP_STEPS:
        return max((step + 1) / WARMUP_STEPS, 1e-8)
    progress = (
        (step - WARMUP_STEPS)
        / max(TOTAL_OPTIMIZER_STEPS - WARMUP_STEPS, 1)
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return 0.05 + 0.95 * cosine


def build_training_state():
    optimizer = torch.optim.AdamW(
        [
            {
                "params": ENCODER_CANDIDATE_PARAMETERS,
                "lr": 1e-5,
                "name": "encoder_last2",
            },
            {
                "params": list(VISUAL_ADAPTER.parameters()),
                "lr": 1e-4,
                "name": "adapter",
            },
            {
                "params": list(SHARED_DECODER.parameters()),
                "lr": 3e-4,
                "name": "decoder",
            },
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_factor,
    )
    scaler = torch.amp.GradScaler(
        "cuda",
        init_scale=1024.0,
    )
    return optimizer, scheduler, scaler


CURRENT_STAGE = set_training_stage(1)
TRAINING_OPTIMIZER, TRAINING_SCHEDULER, TRAINING_SCALER = (
    build_training_state()
)
assert CURRENT_STAGE == "frozen_encoder"
assert sum(
    parameter.numel()
    for parameter in CLIP_ENCODER.parameters()
    if parameter.requires_grad
) == 0
all_ids = []
for group in TRAINING_OPTIMIZER.param_groups:
    all_ids.extend(id(parameter) for parameter in group["params"])
assert len(all_ids) == len(set(all_ids))
assert len(TRAINING_OPTIMIZER.state) == 0

audit_path = AUDIT_DIR / "phase3c_cell05_optimizer_contract.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "stage_1": "frozen_encoder",
        "stage_2": "unfreeze_last_2_blocks_and_post_layernorm",
        "optimizer": "AdamW",
        "optimizer_steps_per_epoch": OPTIMIZER_STEPS_PER_EPOCH,
        "total_optimizer_steps": TOTAL_OPTIMIZER_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "initial_scaler_scale": TRAINING_SCALER.get_scale(),
        "optimizer_state_empty": True,
        "optimizer_step_performed": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Optimizer                 : AdamW")
print("Stage 1                   : frozen encoder, epoch 1–5")
print("Stage 2                   : last 2 blocks, epoch 6–20")
print("Stage 1 encoder trainable : 0")
print("Physical/effective batch  : 16 / 64")
print("Optimizer steps/epoch     :", OPTIMIZER_STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_OPTIMIZER_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("GradScaler initial scale  :", TRAINING_SCALER.get_scale())
print("Optimizer state           : EMPTY")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 10 cells passed")
print("Next                      : run discarded one-batch overfit")

PHASE 3C CELL 5 STATUS
Status                    : PASSED
Optimizer                 : AdamW
Stage 1                   : frozen encoder, epoch 1–5
Stage 2                   : last 2 blocks, epoch 6–20
Stage 1 encoder trainable : 0
Physical/effective batch  : 16 / 64
Optimizer steps/epoch     : 1771
Total optimizer steps     : 35420
Warmup steps              : 1771
GradScaler initial scale  : 1024.0
Optimizer state           : EMPTY
Optimizer step performed  : False
Test Set status           : SEALED
Progress                  : 5 / 10 cells passed
Next                      : run discarded one-batch overfit


## Cell 6 — Discarded one-batch overfit

### Công việc

- lấy 8 ảnh Train, augmentation tắt;
- cache CLIP patch tokens vì encoder đang frozen;
- tối ưu adapter và decoder tối thiểu 50 steps;
- kiểm tra token accuracy và loss reduction;
- phục hồi nguyên vẹn initial weights;
- không lưu overfit weights.

### Thời gian dự kiến

10–60 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Final token accuracy      : >= 0.90
Final loss ratio          : <= 0.25
Initial state restored    : True
Progress                  : 6 / 10 cells passed
```


In [6]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "CLIP_ENCODER",
    "VISUAL_ADAPTER",
    "SHARED_DECODER",
    "TRAIN_DATASET",
    "collate_teacher",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–5: {missing}"

overfit_dataset = ImageBalancedTeacherDataset(SEED, augment=False)
overfit_batch = collate_teacher(
    [overfit_dataset[index] for index in range(8)]
)
overfit_pixels = overfit_batch["pixel_values"].to(device)
overfit_inputs = overfit_batch["decoder_input_ids"].to(device)
overfit_mask = overfit_batch["decoder_attention_mask"].to(device)
overfit_labels = overfit_batch["labels"].to(device)

adapter_initial = copy.deepcopy(VISUAL_ADAPTER.state_dict())
decoder_initial = copy.deepcopy(SHARED_DECODER.state_dict())
CLIP_ENCODER.eval()
with torch.no_grad():
    overfit_patches = CLIP_ENCODER(
        pixel_values=overfit_pixels
    ).last_hidden_state[:, 1:, :]

overfit_optimizer = torch.optim.AdamW(
    [
        {"params": VISUAL_ADAPTER.parameters(), "lr": 1e-3},
        {"params": SHARED_DECODER.parameters(), "lr": 1e-3},
    ],
    weight_decay=0.0,
)


def overfit_evaluate():
    VISUAL_ADAPTER.eval()
    SHARED_DECODER.eval()
    with torch.inference_mode():
        memory = VISUAL_ADAPTER(overfit_patches)
        output = SHARED_DECODER(
            overfit_inputs,
            memory,
            overfit_mask,
        )
        loss_value = F.cross_entropy(
            output.reshape(-1, 10080),
            overfit_labels.reshape(-1),
            ignore_index=0,
        )
        valid = overfit_labels.ne(0)
        accuracy = (
            output.argmax(-1)[valid]
            == overfit_labels[valid]
        ).float().mean()
    return float(loss_value.item()), float(accuracy.item())


initial_loss, initial_accuracy = overfit_evaluate()
steps_completed = 0
for step in range(1, 301):
    VISUAL_ADAPTER.train()
    SHARED_DECODER.train()
    overfit_optimizer.zero_grad(set_to_none=True)
    memory = VISUAL_ADAPTER(overfit_patches)
    output = SHARED_DECODER(
        overfit_inputs,
        memory,
        overfit_mask,
    )
    loss = F.cross_entropy(
        output.reshape(-1, 10080),
        overfit_labels.reshape(-1),
        ignore_index=0,
    )
    assert torch.isfinite(loss)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(VISUAL_ADAPTER.parameters())
        + list(SHARED_DECODER.parameters()),
        1.0,
    )
    overfit_optimizer.step()
    steps_completed = step
    if step >= 50 and step % 10 == 0:
        final_loss, final_accuracy = overfit_evaluate()
        if (
            final_accuracy >= 0.90
            and final_loss / initial_loss <= 0.25
        ):
            break

final_loss, final_accuracy = overfit_evaluate()
loss_ratio = final_loss / initial_loss
assert final_accuracy >= 0.90
assert loss_ratio <= 0.25

VISUAL_ADAPTER.load_state_dict(adapter_initial, strict=True)
SHARED_DECODER.load_state_dict(decoder_initial, strict=True)
restored = all(
    torch.equal(value.cpu(), adapter_initial[key].cpu())
    for key, value in VISUAL_ADAPTER.state_dict().items()
) and all(
    torch.equal(value.cpu(), decoder_initial[key].cpu())
    for key, value in SHARED_DECODER.state_dict().items()
)
assert restored

audit_path = AUDIT_DIR / "phase3c_cell06_one_batch_overfit.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "distinct_train_images": 8,
        "augmentation_disabled": True,
        "encoder_frozen_and_cached": True,
        "initial_loss": initial_loss,
        "final_loss": final_loss,
        "loss_ratio": loss_ratio,
        "initial_token_accuracy": initial_accuracy,
        "final_token_accuracy": final_accuracy,
        "steps_completed": steps_completed,
        "initial_state_restored": restored,
        "overfit_weights_saved": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

del overfit_optimizer
print("=" * 72)
print("PHASE 3C CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Distinct Train images     : 8")
print("Initial loss              :", f"{initial_loss:.8f}")
print("Final loss                :", f"{final_loss:.8f}")
print("Final loss ratio          :", f"{loss_ratio:.8f}")
print("Final token accuracy      :", f"{final_accuracy:.6f}")
print("Completed steps           :", steps_completed)
print("Initial state restored    :", restored)
print("Overfit weights saved     : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 10 cells passed")
print("Next                      : run discarded FP16 optimizer step")

PHASE 3C CELL 6 STATUS
Status                    : PASSED
Distinct Train images     : 8
Initial loss              : 10.52593327
Final loss                : 0.00155744
Final loss ratio          : 0.00014796
Final token accuracy      : 1.000000
Completed steps           : 50
Initial state restored    : True
Overfit weights saved     : False
Test Set status           : SEALED
Progress                  : 6 / 10 cells passed
Next                      : run discarded FP16 optimizer step


## Cell 7 — Discarded FP16 optimizer-step preflight

### Công việc

- chạy 4 microbatches, effective batch 64;
- áp dụng token-balanced gradient accumulation;
- xác minh adapter/decoder cập nhật;
- kiểm tra overflow và gradient hữu hạn;
- phục hồi initial weights và optimizer state.

### Thời gian dự kiến

15–90 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Preflight optimizer steps : 1
Images                    : 64
FP16 overflow             : False
Initial state restored    : True
Progress                  : 7 / 10 cells passed
```


In [7]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "TRAIN_DATASET",
    "make_loader",
    "CLIP_ENCODER",
    "VISUAL_ADAPTER",
    "SHARED_DECODER",
    "build_training_state",
    "set_training_stage",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–6: {missing}"

BACKWARD_DIVISOR = 1024.0


def restore_light_initial():
    VISUAL_ADAPTER.load_state_dict(
        LIGHT_CHECKPOINT["adapter_state_dict"],
        strict=True,
    )
    SHARED_DECODER.load_state_dict(
        LIGHT_CHECKPOINT["decoder_state_dict"],
        strict=True,
    )
    set_training_stage(1)


def move_batch(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if torch.is_tensor(value)
            else value
        )
        for key, value in batch.items()
    }


def run_optimizer_step(batches, optimizer, scheduler, scaler):
    optimizer.zero_grad(set_to_none=True)
    CLIP_ENCODER.eval()
    VISUAL_ADAPTER.train()
    SHARED_DECODER.train()
    total_loss_sum = 0.0
    total_tokens = 0
    scale_before = scaler.get_scale()
    for raw_batch in batches:
        batch = move_batch(raw_batch)
        with torch.no_grad():
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                patches = CLIP_ENCODER(
                    pixel_values=batch["pixel_values"]
                ).last_hidden_state[:, 1:, :]
        with torch.amp.autocast("cuda", dtype=torch.float16):
            memory = VISUAL_ADAPTER(patches)
            logits = SHARED_DECODER(
                batch["decoder_input_ids"],
                memory,
                batch["decoder_attention_mask"],
            )
            loss_sum = F.cross_entropy(
                logits.reshape(-1, 10080),
                batch["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
        valid_tokens = int(batch["labels"].ne(0).sum().item())
        assert valid_tokens > 0 and torch.isfinite(loss_sum)
        scaler.scale(loss_sum / BACKWARD_DIVISOR).backward()
        total_loss_sum += float(loss_sum.detach().item())
        total_tokens += valid_tokens
    scaler.unscale_(optimizer)
    correction = BACKWARD_DIVISOR / total_tokens
    for group in optimizer.param_groups:
        for parameter in group["params"]:
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
    trainable = [
        parameter
        for group in optimizer.param_groups
        for parameter in group["params"]
        if parameter.grad is not None
    ]
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        trainable, 1.0
    )
    assert torch.isfinite(gradient_norm)
    scaler.step(optimizer)
    scaler.update()
    overflow = scaler.get_scale() < scale_before
    if not overflow:
        scheduler.step()
    return {
        "mean_token_loss": total_loss_sum / total_tokens,
        "valid_tokens": total_tokens,
        "gradient_norm": float(gradient_norm.item()),
        "overflow": overflow,
    }


restore_light_initial()
preflight_adapter_before = copy.deepcopy(
    VISUAL_ADAPTER.state_dict()
)
preflight_decoder_before = copy.deepcopy(
    SHARED_DECODER.state_dict()
)
optimizer, scheduler, scaler = build_training_state()
TRAIN_DATASET.set_epoch(0)
loader = make_loader(TRAIN_DATASET)
iterator = iter(loader)
batches = [next(iterator) for _ in range(4)]
result = run_optimizer_step(
    batches, optimizer, scheduler, scaler
)
assert result["overflow"] is False
adapter_changed = any(
    not torch.equal(
        value.cpu(),
        preflight_adapter_before[key].cpu(),
    )
    for key, value in VISUAL_ADAPTER.state_dict().items()
)
decoder_changed = any(
    not torch.equal(
        value.cpu(),
        preflight_decoder_before[key].cpu(),
    )
    for key, value in SHARED_DECODER.state_dict().items()
)
assert adapter_changed and decoder_changed

restore_light_initial()
optimizer, scheduler, scaler = build_training_state()
restored = (
    len(optimizer.state) == 0
    and all(
        torch.equal(
            value.cpu(),
            LIGHT_CHECKPOINT["adapter_state_dict"][key],
        )
        for key, value in VISUAL_ADAPTER.state_dict().items()
    )
    and all(
        torch.equal(
            value.cpu(),
            LIGHT_CHECKPOINT["decoder_state_dict"][key],
        )
        for key, value in SHARED_DECODER.state_dict().items()
    )
)
assert restored

audit_path = AUDIT_DIR / "phase3c_cell07_fp16_preflight.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell07_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "optimizer_steps": 1,
        "microbatches": 4,
        "images": 64,
        **result,
        "adapter_changed": adapter_changed,
        "decoder_changed": decoder_changed,
        "initial_state_restored": restored,
        "temporary_weights_saved": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Preflight optimizer steps : 1")
print("Microbatches              : 4")
print("Images                    : 64")
print("Mean token CE             :", f"{result['mean_token_loss']:.8f}")
print("Gradient norm             :", f"{result['gradient_norm']:.8f}")
print("FP16 overflow             :", result["overflow"])
print("Adapter updated           :", adapter_changed)
print("Decoder updated           :", decoder_changed)
print("Initial state restored    :", restored)
print("Test Set status           : SEALED")
print("Progress                  : 7 / 10 cells passed")
print("Next                      : exact-resume equivalence")

PHASE 3C CELL 7 STATUS
Status                    : PASSED
Preflight optimizer steps : 1
Microbatches              : 4
Images                    : 64
Mean token CE             : 10.53481419
Gradient norm             : 9.20613956
FP16 overflow             : False
Adapter updated           : True
Decoder updated           : True
Initial state restored    : True
Test Set status           : SEALED
Progress                  : 7 / 10 cells passed
Next                      : exact-resume equivalence


## Cell 8 — Exact-resume equivalence

### Công việc

- nhánh liên tục chạy hai optimizer steps;
- nhánh resume chạy một step, save/load, rồi chạy step thứ hai;
- so sánh adapter, decoder, optimizer, scheduler và GradScaler;
- checkpoint tạm được xóa.

### Thời gian dự kiến

30–180 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Loss sequence exact       : True
Model state exact         : True
Optimizer state exact     : True
Temporary checkpoint      : DELETED
Progress                  : 8 / 10 cells passed
```


In [8]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "restore_light_initial",
    "build_training_state",
    "run_optimizer_step",
    "TRAIN_DATASET",
    "make_loader",
    "CHECKPOINT_DIR",
    "AUDIT_DIR",
    "set_deterministic",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–7: {missing}"


def clone_cpu_state(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def tensors_exact(left, right):
    return (
        left.keys() == right.keys()
        and all(torch.equal(left[key], right[key]) for key in left)
    )


def nested_exact(left, right):
    if torch.is_tensor(left):
        return torch.is_tensor(right) and torch.equal(
            left.cpu(), right.cpu()
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and left.keys() == right.keys()
            and all(nested_exact(left[key], right[key]) for key in left)
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(nested_exact(a, b) for a, b in zip(left, right))
        )
    return left == right


TRAIN_DATASET.set_epoch(0)
resume_loader = make_loader(TRAIN_DATASET)
resume_iterator = iter(resume_loader)
resume_batches = [next(resume_iterator) for _ in range(8)]

set_deterministic(SEED + 800)
restore_light_initial()
continuous_optimizer, continuous_scheduler, continuous_scaler = (
    build_training_state()
)
continuous_results = []
for step_index in range(2):
    batch_slice = resume_batches[
        step_index * 4 : (step_index + 1) * 4
    ]
    continuous_results.append(
        run_optimizer_step(
            batch_slice,
            continuous_optimizer,
            continuous_scheduler,
            continuous_scaler,
        )
    )
continuous_adapter = clone_cpu_state(VISUAL_ADAPTER)
continuous_decoder = clone_cpu_state(SHARED_DECODER)
continuous_optimizer_state = copy.deepcopy(
    continuous_optimizer.state_dict()
)
continuous_scheduler_state = copy.deepcopy(
    continuous_scheduler.state_dict()
)
continuous_scaler_state = copy.deepcopy(
    continuous_scaler.state_dict()
)

set_deterministic(SEED + 800)
restore_light_initial()
resume_optimizer, resume_scheduler, resume_scaler = (
    build_training_state()
)
resume_results = [
    run_optimizer_step(
        resume_batches[:4],
        resume_optimizer,
        resume_scheduler,
        resume_scaler,
    )
]
temporary_checkpoint = (
    CHECKPOINT_DIR / "temporary_exact_resume.pt"
)
torch.save(
    {
        "adapter": clone_cpu_state(VISUAL_ADAPTER),
        "decoder": clone_cpu_state(SHARED_DECODER),
        "optimizer": resume_optimizer.state_dict(),
        "scheduler": resume_scheduler.state_dict(),
        "scaler": resume_scaler.state_dict(),
        "cpu_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    },
    temporary_checkpoint,
)

restore_light_initial()
resume_optimizer, resume_scheduler, resume_scaler = (
    build_training_state()
)
checkpoint = torch.load(
    temporary_checkpoint,
    map_location="cpu",
    weights_only=True,
)
VISUAL_ADAPTER.load_state_dict(checkpoint["adapter"], strict=True)
SHARED_DECODER.load_state_dict(checkpoint["decoder"], strict=True)
resume_optimizer.load_state_dict(checkpoint["optimizer"])
resume_scheduler.load_state_dict(checkpoint["scheduler"])
resume_scaler.load_state_dict(checkpoint["scaler"])
torch.set_rng_state(checkpoint["cpu_rng"])
torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
resume_results.append(
    run_optimizer_step(
        resume_batches[4:],
        resume_optimizer,
        resume_scheduler,
        resume_scaler,
    )
)

resume_adapter = clone_cpu_state(VISUAL_ADAPTER)
resume_decoder = clone_cpu_state(SHARED_DECODER)
loss_sequence_exact = all(
    a["mean_token_loss"] == b["mean_token_loss"]
    and a["gradient_norm"] == b["gradient_norm"]
    and a["overflow"] == b["overflow"]
    for a, b in zip(continuous_results, resume_results)
)
adapter_exact = tensors_exact(
    continuous_adapter, resume_adapter
)
decoder_exact = tensors_exact(
    continuous_decoder, resume_decoder
)
optimizer_exact = nested_exact(
    continuous_optimizer_state,
    resume_optimizer.state_dict(),
)
scheduler_exact = nested_exact(
    continuous_scheduler_state,
    resume_scheduler.state_dict(),
)
scaler_exact = nested_exact(
    continuous_scaler_state,
    resume_scaler.state_dict(),
)
assert all(
    [
        loss_sequence_exact,
        adapter_exact,
        decoder_exact,
        optimizer_exact,
        scheduler_exact,
        scaler_exact,
    ]
)
temporary_checkpoint.unlink()
assert not temporary_checkpoint.exists()
restore_light_initial()

audit_path = AUDIT_DIR / "phase3c_cell08_exact_resume.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell08_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "continuous_optimizer_steps": 2,
        "resume_pattern": "1_save_load_1",
        "loss_sequence_exact": loss_sequence_exact,
        "adapter_state_exact": adapter_exact,
        "decoder_state_exact": decoder_exact,
        "optimizer_state_exact": optimizer_exact,
        "scheduler_state_exact": scheduler_exact,
        "gradscaler_state_exact": scaler_exact,
        "temporary_checkpoint_deleted": True,
        "initial_state_restored": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Continuous steps          : 2")
print("Resume pattern            : 1 + save/load + 1")
print("Loss sequence exact       :", loss_sequence_exact)
print("Adapter state exact       :", adapter_exact)
print("Decoder state exact       :", decoder_exact)
print("Optimizer state exact     :", optimizer_exact)
print("Scheduler state exact     :", scheduler_exact)
print("GradScaler state exact    :", scaler_exact)
print("Temporary checkpoint      : DELETED")
print("Initial state restored    : True")
print("Test Set status           : SEALED")
print("Progress                  : 8 / 10 cells passed")
print("Next                      : discarded throughput pilot")

PHASE 3C CELL 8 STATUS
Status                    : PASSED
Continuous steps          : 2
Resume pattern            : 1 + save/load + 1
Loss sequence exact       : True
Adapter state exact       : True
Decoder state exact       : True
Optimizer state exact     : True
Scheduler state exact     : True
GradScaler state exact    : True
Temporary checkpoint      : DELETED
Initial state restored    : True
Test Set status           : SEALED
Progress                  : 8 / 10 cells passed
Next                      : discarded throughput pilot


## Cell 9 — Discarded throughput pilot

### Công việc

- chạy 20 optimizer steps, 1.280 ảnh;
- dùng đúng FP16 và accumulation 4;
- đo giây/step, ước lượng phút/epoch và peak GPU memory;
- kiểm tra overflow/OOM;
- xóa toàn bộ pilot weights bằng cách phục hồi initial state.

### Thời gian dự kiến

2–15 phút trên Tesla T4.

### Đầu ra dự kiến

```text
Status                    : PASSED
Pilot optimizer steps     : 20
Pilot images              : 1280
FP16 overflow steps       : 0
Initial state restored    : True
Progress                  : 9 / 10 cells passed
```


In [9]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "restore_light_initial",
    "build_training_state",
    "run_optimizer_step",
    "TRAIN_DATASET",
    "make_loader",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–8: {missing}"

PILOT_OPTIMIZER_STEPS = 20
restore_light_initial()
pilot_optimizer, pilot_scheduler, pilot_scaler = (
    build_training_state()
)
TRAIN_DATASET.set_epoch(0)
pilot_loader = make_loader(TRAIN_DATASET)
pilot_iterator = iter(pilot_loader)
torch.cuda.reset_peak_memory_stats()
pilot_started = time.perf_counter()
pilot_rows = []
overflow_steps = 0
for optimizer_step in range(1, PILOT_OPTIMIZER_STEPS + 1):
    batches = [next(pilot_iterator) for _ in range(4)]
    result = run_optimizer_step(
        batches,
        pilot_optimizer,
        pilot_scheduler,
        pilot_scaler,
    )
    overflow_steps += int(result["overflow"])
    pilot_rows.append(
        {
            "optimizer_step": optimizer_step,
            **result,
        }
    )
    if optimizer_step in {1, 5, 10, 15, 20}:
        print(
            f"step={optimizer_step:03d} | "
            f"loss={result['mean_token_loss']:.6f} | "
            f"grad={result['gradient_norm']:.4f} | "
            f"overflow={result['overflow']}"
        )
pilot_seconds = time.perf_counter() - pilot_started
seconds_per_step = pilot_seconds / PILOT_OPTIMIZER_STEPS
estimated_minutes_per_epoch = (
    seconds_per_step * OPTIMIZER_STEPS_PER_EPOCH / 60.0
)
peak_gpu_memory_mib = (
    torch.cuda.max_memory_allocated() / (1024 ** 2)
)
assert overflow_steps == 0

pilot_log_path = LOG_DIR / "teacher_a_20_step_pilot.json"
write_json(
    pilot_log_path,
    {
        "status": "passed",
        "optimizer_steps": PILOT_OPTIMIZER_STEPS,
        "images": PILOT_OPTIMIZER_STEPS * 64,
        "rows": pilot_rows,
        "seconds_per_optimizer_step": seconds_per_step,
        "estimated_minutes_per_epoch": estimated_minutes_per_epoch,
        "peak_gpu_memory_mib": peak_gpu_memory_mib,
        "overflow_steps": overflow_steps,
        "weights_retained": False,
    },
)

restore_light_initial()
initial_restored = all(
    torch.equal(
        value.cpu(),
        LIGHT_CHECKPOINT["adapter_state_dict"][key],
    )
    for key, value in VISUAL_ADAPTER.state_dict().items()
) and all(
    torch.equal(
        value.cpu(),
        LIGHT_CHECKPOINT["decoder_state_dict"][key],
    )
    for key, value in SHARED_DECODER.state_dict().items()
)
assert initial_restored

audit_path = AUDIT_DIR / "phase3c_cell09_throughput_pilot.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3c_cell09_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "optimizer_steps": PILOT_OPTIMIZER_STEPS,
        "microbatches": PILOT_OPTIMIZER_STEPS * 4,
        "images": PILOT_OPTIMIZER_STEPS * 64,
        "overflow_steps": overflow_steps,
        "seconds_per_optimizer_step": seconds_per_step,
        "estimated_minutes_per_epoch": estimated_minutes_per_epoch,
        "peak_gpu_memory_mib": peak_gpu_memory_mib,
        "initial_state_restored": initial_restored,
        "pilot_weights_retained": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3C CELL 9 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Pilot optimizer steps     :", PILOT_OPTIMIZER_STEPS)
print("Pilot microbatches        :", PILOT_OPTIMIZER_STEPS * 4)
print("Pilot images              :", PILOT_OPTIMIZER_STEPS * 64)
print("FP16 overflow steps       :", overflow_steps)
print("Seconds/optimizer step    :", f"{seconds_per_step:.4f}")
print("Estimated train min/epoch :", f"{estimated_minutes_per_epoch:.2f}")
print("Peak GPU memory           :", f"{peak_gpu_memory_mib:.2f} MiB")
print("Initial state restored    :", initial_restored)
print("Pilot weights retained    : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 9 / 10 cells passed")
print("Next                      : package full-training readiness")

step=001 | loss=10.585642 | grad=9.3977 | overflow=False
step=005 | loss=10.566849 | grad=9.1246 | overflow=False
step=010 | loss=10.474259 | grad=9.5045 | overflow=False
step=015 | loss=10.503844 | grad=9.1505 | overflow=False
step=020 | loss=10.468724 | grad=9.2472 | overflow=False
PHASE 3C CELL 9 STATUS
Status                    : PASSED
Pilot optimizer steps     : 20
Pilot microbatches        : 80
Pilot images              : 1280
FP16 overflow steps       : 0
Seconds/optimizer step    : 0.6356
Estimated train min/epoch : 18.76
Peak GPU memory           : 749.11 MiB
Initial state restored    : True
Pilot weights retained    : False
Full training started     : False
Test Set status           : SEALED
Progress                  : 9 / 10 cells passed
Next                      : package full-training readiness


## Cell 10 — Package và full-training READY marker

### Công việc

- verify audit Cell 1–9;
- xác nhận overfit, FP16, resume và throughput gates;
- tạo manifest CSV/JSON/SHA256SUMS;
- tạo `PHASE3_TEACHER_A_FULL_TRAINING_READY.json`;
- cho phép tạo Phase 03D full training.

### Thời gian dự kiến

2–20 giây.

### Đầu ra bắt buộc

```text
Status                    : PASSED
Teacher A training engine : READY
One-batch overfit         : PASSED
Exact resume              : PASSED
Throughput pilot          : PASSED
Full training started     : False
Test Set status           : SEALED
Progress                  : 10 / 10 cells passed
```


In [10]:
CELL_STARTED_AT = time.perf_counter()
required_audits = [
    AUDIT_DIR / "phase3c_cell01_phase3b_environment.json",
    AUDIT_DIR / "phase3c_cell02_data_indexes.json",
    AUDIT_DIR / "phase3c_cell03_dataloader.json",
    AUDIT_DIR / "phase3c_cell04_model_load.json",
    AUDIT_DIR / "phase3c_cell05_optimizer_contract.json",
    AUDIT_DIR / "phase3c_cell06_one_batch_overfit.json",
    AUDIT_DIR / "phase3c_cell07_fp16_preflight.json",
    AUDIT_DIR / "phase3c_cell08_exact_resume.json",
    AUDIT_DIR / "phase3c_cell09_throughput_pilot.json",
]
for audit in required_audits:
    assert audit.is_file(), f"Thiếu audit: {audit}"
    assert read_json(audit)["status"] == "passed"

overfit_audit = read_json(required_audits[5])
resume_audit = read_json(required_audits[7])
pilot_audit = read_json(required_audits[8])
assert overfit_audit["final_token_accuracy"] >= 0.90
assert overfit_audit["loss_ratio"] <= 0.25
assert resume_audit["loss_sequence_exact"] is True
assert resume_audit["optimizer_state_exact"] is True
assert pilot_audit["overflow_steps"] == 0
assert pilot_audit["initial_state_restored"] is True

MANIFEST_CSV = MANIFEST_DIR / "teacher_a_training_readiness_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "teacher_a_training_readiness_manifest.json"
SHA256SUMS = MANIFEST_DIR / "teacher_a_training_readiness_SHA256SUMS.txt"
READY_PATH = OUTPUT_ROOT / "PHASE3_TEACHER_A_FULL_TRAINING_READY.json"
excluded = {
    MANIFEST_CSV.resolve(),
    MANIFEST_JSON.resolve(),
    SHA256SUMS.resolve(),
    READY_PATH.resolve(),
}
rows = []
for artifact in sorted(
    OUTPUT_ROOT.rglob("*"),
    key=lambda value: value.as_posix(),
):
    if not artifact.is_file() or artifact.resolve() in excluded:
        continue
    if artifact.name.endswith(".tmp"):
        raise AssertionError(f"Temporary artifact: {artifact}")
    rows.append(
        {
            "relative_path": artifact.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": artifact.stat().st_size,
            "sha256": sha256_file(artifact),
        }
    )
with open(
    MANIFEST_CSV,
    "w",
    encoding="utf-8",
    newline="",
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase3_teacher_a_training_readiness_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(rows),
        "total_size_bytes": sum(row["size_bytes"] for row in rows),
        "artifacts": rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
SHA256SUMS.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in rows
    ),
    encoding="utf-8",
)
with open(
    MANIFEST_CSV,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    verified_rows = list(csv.DictReader(source))
for row in verified_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

ready_payload = {
    "marker_version": "phase3_teacher_a_full_training_ready_v1",
    "created_at_utc": now_utc(),
    "status": "ready",
    "teacher_id": "A",
    "teacher": "CLIP ViT-B/32 + shared caption decoder",
    "development_seed": 42,
    "training_contract_sha256": sha256_file(TRAINING_CONTRACT_PATH),
    "training_data_contract": {
        "relative_path": (
            CONFIG_DIR / "teacher_a_training_data_contract.json"
        ).relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(
            CONFIG_DIR / "teacher_a_training_data_contract.json"
        ),
    },
    "gates": {
        "one_batch_overfit": "passed",
        "fp16_optimizer_step": "passed",
        "exact_resume": "passed",
        "throughput_pilot": "passed",
        "initial_state_restored": True,
    },
    "pilot": {
        "optimizer_steps": pilot_audit["optimizer_steps"],
        "seconds_per_optimizer_step": pilot_audit[
            "seconds_per_optimizer_step"
        ],
        "estimated_minutes_per_epoch": pilot_audit[
            "estimated_minutes_per_epoch"
        ],
        "peak_gpu_memory_mib": pilot_audit[
            "peak_gpu_memory_mib"
        ],
    },
    "manifests": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV),
        },
        "json": {
            "relative_path": MANIFEST_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS),
        },
    },
    "authorized_next_action": (
        "Create Phase 03D Teacher A seed-42 full training notebook. "
        "Select checkpoints using Validation CIDEr-D only."
    ),
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
}
write_json(READY_PATH, ready_payload)

elapsed = time.perf_counter() - CELL_STARTED_AT
print("=" * 72)
print("PHASE 3C CELL 10 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A training engine : READY")
print("Audit cells               : 9 / 9 PASSED")
print("One-batch overfit         : PASSED")
print("FP16 optimizer step       : PASSED")
print("Exact resume              : PASSED")
print("Throughput pilot          : PASSED")
print("Manifest artifacts        :", len(rows))
print("Manifest verification     : ALL PASSED")
print("Initial state restored    : True")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("READY marker              :", READY_PATH)
print("READY SHA-256             :", sha256_file(READY_PATH))
print("Progress                  : 10 / 10 cells passed")
print("Elapsed                   :", round(elapsed, 2), "giây")
print("Next                      : Save Version, then Phase 03D full training")

PHASE 3C CELL 10 STATUS
Status                    : PASSED
Teacher A training engine : READY
Audit cells               : 9 / 9 PASSED
One-batch overfit         : PASSED
FP16 optimizer step       : PASSED
Exact resume              : PASSED
Throughput pilot          : PASSED
Manifest artifacts        : 11
Manifest verification     : ALL PASSED
Initial state restored    : True
Full training started     : False
Test Set status           : SEALED
Test Set used             : False
Flickr data used          : False
READY marker              : /kaggle/working//phase3_teacher_a_training_readiness/PHASE3_TEACHER_A_FULL_TRAINING_READY.json
READY SHA-256             : 57e707c3c2a79be50e7c5c63e763f98faf20be5fef0f80e769a9d303458d994f
Progress                  : 10 / 10 cells passed
Elapsed                   : 0.02 giây
Next                      : Save Version, then Phase 03D full training
